# Capstone - Small Business Finance Assistant (Solution)
**Day 3 - Session 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/capstone/solution/capstone.ipynb)

**Goal:** Build a retrieval-augmented assistant that answers small-business
finance questions over a mock QuickBooks and TurboTax help knowledge base,
grounded in the retrieved help articles. Then extend it with agentic tools,
an LLM-as-a-judge evaluation, a retrieval comparison, and a LoRA-trained
ticket router.

All data in `capstone/data/` is synthetic mock data. This solution has the
full working code. Retrieval and fine-tuning run offline; generation and the
judge call an LLM API through the shared client. Runs on CPU; no GPU needed.

In [ ]:
# Scoped install for this capstone (safe to re-run; preinstalled on Colab).
%pip install -q sentence-transformers faiss-cpu litellm smolagents peft accelerate transformers torch python-dotenv

## Setup: shared client and data path
This finds the `shared/` package and the `capstone/data/` folder locally, or
clones the companion repo when running on Colab. It also makes sure an API key
is set.

In [ ]:
import os
import subprocess
import sys

import torch  # noqa: F401  (import torch before faiss to avoid an OpenMP clash on macOS)


def _find_repo_root():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, "shared")):
            return d
        d = os.path.dirname(d)
    return None


root = _find_repo_root()
if root is None:  # On Colab: clone the companion repo for shared/ and data.
    if not os.path.isdir("gen-ai-companion-intuite"):
        subprocess.run(
            ["git", "clone", "-q",
             "https://github.com/kpassoubady/gen-ai-companion-intuite.git"],
            check=True,
        )
    root = "gen-ai-companion-intuite"
sys.path.insert(0, root)
DATA_DIR = os.path.join(root, "capstone", "data")

from shared import get_completion, get_model, show_config

if not any(os.environ.get(k) for k in ("OPENAI_API_KEY", "GEMINI_API_KEY",
                                       "ANTHROPIC_API_KEY", "AZURE_API_KEY")):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
    os.environ.setdefault("LLM_PROVIDER", "openai")

show_config()

## 1. Load the knowledge base
Read every Markdown help article from `capstone/data/`. Each file is one
source document the assistant is allowed to use.

In [ ]:
import glob

documents = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, "*.md"))):
    with open(path, encoding="utf-8") as f:
        documents.append({"source": os.path.basename(path), "text": f.read()})
print(f"Loaded {len(documents)} help articles:",
      ", ".join(d["source"] for d in documents))

## 2. Chunk, embed, and index
Split each article into overlapping chunks, embed them with a small local
model, and index the vectors with FAISS. Normalized vectors make inner-product
search equal to cosine similarity.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


def chunk_text(text, chunk_size=500, overlap=60):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks


def build_index(docs, model, chunk_size=500, overlap=60):
    chunks = []
    for doc in docs:
        for piece in chunk_text(doc["text"], chunk_size, overlap):
            chunks.append({"source": doc["source"], "text": piece})
    vecs = np.array(model.encode([c["text"] for c in chunks],
                                 normalize_embeddings=True), dtype="float32")
    idx = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs)
    return chunks, idx


embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunks, index = build_index(documents, embedder)
print(f"Indexed {index.ntotal} chunks.")

## 3. Retrieve and answer, grounded
Retrieve the top chunks for a question, then answer using only that context.
If the context does not contain the answer, the assistant says so and cites
the source article otherwise.

In [ ]:
SYSTEM = (
    "You are a small-business finance assistant. Answer using ONLY the provided "
    "context from QuickBooks and TurboTax help articles. If the context does not "
    "contain the answer, say you do not have that information. Cite the source "
    "file in brackets, for example [quickbooks_invoicing.md]."
)


def retrieve(query, model, idx, chunk_list, k=3):
    q = np.array(model.encode([query], normalize_embeddings=True), dtype="float32")
    scores, ids = idx.search(q, k)
    hits = []
    for score, i in zip(scores[0], ids[0]):
        hit = dict(chunk_list[i])
        hit["score"] = float(score)
        hits.append(hit)
    return hits


def rag_answer(query, k=3):
    hits = retrieve(query, embedder, index, chunks, k)
    context = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    return get_completion(messages, tier="mini", temperature=0.0), hits


DEMO_QUESTIONS = [
    "What is the default payment term on a QuickBooks invoice?",
    "How much does QuickBooks Payroll Core cost?",
    "Can I deduct a home office, and how much?",
    "Does QuickBooks integrate with Salesforce?",  # not in the KB
]
for q in DEMO_QUESTIONS:
    ans, _ = rag_answer(q)
    print(f"\nQ: {q}\nA: {ans}")

## Stretch 1: agentic RAG with a computation tool
RAG retrieves facts but does not compute. We give an agent two tools: one to
search the knowledge base and one to compute an invoice total with sales tax.
The agent decides when to search and when to calculate.

In [ ]:
from smolagents import CodeAgent, LiteLLMModel, tool

TAX_RATES = {"CA": 7.25, "NY": 4.0, "TX": 6.25, "WA": 6.5, "FL": 6.0}


@tool
def search_help(query: str) -> str:
    """Search the QuickBooks and TurboTax help knowledge base.

    Args:
        query: The natural-language search query.
    """
    hits = retrieve(query, embedder, index, chunks, k=3)
    return "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)


@tool
def invoice_total(subtotal: float, state_code: str) -> float:
    """Compute an invoice total including state sales tax.

    Args:
        subtotal: The pre-tax invoice subtotal in dollars.
        state_code: Two-letter state code, for example CA or NY.
    """
    rate = TAX_RATES.get(state_code.upper(), 0.0)
    return round(subtotal * (1 + rate / 100.0), 2)


agent = CodeAgent(tools=[search_help, invoice_total],
                  model=LiteLLMModel(model_id=get_model("mini")), max_steps=6)
print(agent.run(
    "A customer in California owes 1200 dollars before tax. Using the course "
    "sales tax rate, what is the invoice total including sales tax?"
))

## Stretch 2: evaluate with an LLM-as-a-judge
We score the assistant on the golden question and answer pairs in
`golden_qa.json`. The judge checks whether each answer is faithful to the
retrieved context and matches the expected answer.

In [ ]:
import json

with open(os.path.join(DATA_DIR, "golden_qa.json"), encoding="utf-8") as f:
    golden = json.load(f)

JUDGE_SYSTEM = (
    "You are a strict evaluator. Given a question, the retrieved context, the "
    "model answer, and the expected answer, return ONLY JSON like "
    '{"faithful": true, "correct": true}. faithful means the answer is supported '
    "by the context; correct means it matches the expected answer."
)


def judge(question, context, answer, expected):
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user", "content": (
            f"Question: {question}\nContext:\n{context}\n"
            f"Model answer: {answer}\nExpected: {expected}")},
    ]
    raw = get_completion(messages, tier="mini", temperature=0.0)
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    return json.loads(raw)


passed = 0
for case in golden:
    ans, hits = rag_answer(case["question"])
    context = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    verdict = judge(case["question"], context, ans, case["expected"])
    ok = bool(verdict.get("faithful") and verdict.get("correct"))
    passed += int(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {case['question']}")

print(f"\nScore: {passed}/{len(golden)} faithful and correct.")

## Stretch 3: compare chunking strategies
Retrieval quality depends on how you chunk. We rebuild the index with smaller
chunks and compare which source articles each setting retrieves for the same
questions.

In [ ]:
small_chunks, small_index = build_index(documents, embedder, chunk_size=200, overlap=30)
print(f"Default chunks: {len(chunks)}   small chunks: {len(small_chunks)}\n")

for q in [c["question"] for c in golden[:3]]:
    default_src = retrieve(q, embedder, index, chunks, k=1)[0]["source"]
    small_src = retrieve(q, embedder, small_index, small_chunks, k=1)[0]["source"]
    match = "same" if default_src == small_src else "different"
    print(f"[{match}] {q}\n   default -> {default_src}   small -> {small_src}")

## Stretch 4: LoRA-train a support-ticket router
We adapt the LoRA idea from Lab 3.2 to a classifier: a DistilBERT encoder with
a LoRA adapter learns to route tickets into billing, payroll, tax_filing, or
integrations. Only the adapter and the head train.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments)
from torch.utils.data import Dataset

with open(os.path.join(DATA_DIR, "support_tickets.json"), encoding="utf-8") as f:
    tickets = json.load(f)

LABELS = ["billing", "payroll", "tax_filing", "integrations"]
label2id = {name: i for i, name in enumerate(LABELS)}
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")


class TicketDataset(Dataset):
    def __init__(self, rows):
        self.enc = tok([r["text"] for r in rows], padding=True,
                       truncation=True, max_length=48)
        self.labels = [label2id[r["label"]] for r in rows]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item


clf = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(LABELS),
    id2label={i: n for n, i in label2id.items()}, label2id=label2id)
clf = get_peft_model(clf, LoraConfig(
    r=8, lora_alpha=16, target_modules=["q_lin", "v_lin"],
    task_type="SEQ_CLS"))
clf.print_trainable_parameters()

Train the router for a handful of epochs on the 16 example tickets, then route
the held-out test tickets.

In [ ]:
trainer = Trainer(
    model=clf,
    args=TrainingArguments(output_dir="ticket-router", num_train_epochs=30,
                           per_device_train_batch_size=8, learning_rate=5e-4,
                           logging_steps=20, report_to="none", use_cpu=True,
                           seed=42),
    train_dataset=TicketDataset(tickets["train"]),
)
trainer.train()

clf.eval()
correct = 0
for row in tickets["test"]:
    enc = tok(row["text"], return_tensors="pt", truncation=True, max_length=48)
    with torch.no_grad():
        pred = clf(**enc).logits.argmax(-1).item()
    routed = clf.config.id2label[pred]
    correct += int(routed == row["label"])
    print(f"[{routed:>12} | true {row['label']:>12}]  {row['text']}")
print(f"\nRouting accuracy: {correct}/{len(tickets['test'])}")

## Expected output
- The base assistant answers the finance questions with a source citation and
  declines the Salesforce question because the fact is not in the KB.
- The agentic stretch searches the KB and computes a 1200 dollar California
  invoice total of 1287.00.
- The judge marks the golden Q&A set faithful and correct (about 5/5).
- The chunking comparison shows which questions retrieve the same source under
  both chunk sizes.
- The LoRA ticket router trains only a small adapter and routes most or all of
  the four held-out tickets correctly.